#  PCA y clustering: el nivel del mar en la costa

**Analítica de datos aplicada a la Ingeniería Civil**

**Caso:** anomalía del nivel del mar (SLA) en 171 puntos de costa · 192 meses, de enero de 2000 a diciembre de 2015

> 🎯 **Mensaje clave:** sin variable respuesta también se aprende. El PCA resume muchas series en unos pocos **patrones** con significado físico, y k-means agrupa los mapas en **situaciones típicas**.

### Cómo trabajar con este notebook

- Ejecuta las celdas **en orden**, de arriba abajo (`Shift + Enter`).
- Las celdas marcadas con **✏️** tienen un hueco `___` que debes rellenar. Si no lo rellenas, la celda dará error.
- Los bloques **💡 Preguntas** son para pensar y comentar en clase: lo importante es **interpretar**, no escribir código.

| Bloque | Qué hacemos en este notebook |
|:--:|---|
| 1 | Intuición: PCA en 2D |
| 2 | Los datos de nivel del mar (SLA) |
| 3 | PCA: ¿cuántas componentes? |
| 4 | Patrones espaciales (EOF) y series temporales (PC) |
| 5 | Reconstruir los mapas con pocas componentes |
| 6 | K-means sobre las componentes: ¿cuántos grupos? |
| 7 | Interpretar los grupos: mapas medios y cuándo aparecen |
| 8 | *Ampliación (si sobra tiempo):* quitar el ciclo estacional y la tendencia |

### Imports y configuración básica

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
from scipy.io import loadmat

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Colores que usaremos en todo el notebook
AZUL = "#4C72B0"
NARANJA = "#DD8452"
ROJO = "#C44E52"
COLORES_GRUPOS = plt.cm.tab10.colors   # un color por grupo de k-means
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

plt.rcParams["font.size"] = 12


def estilo(ax):
    """Estética común de las gráficas: sin bordes arriba y a la derecha, rejilla suave."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.25)

---
## 1. Intuición: PCA en 2D

Generamos 500 puntos de dos variables **correlacionadas** y dibujamos sus dos componentes principales. Cada flecha mide dos desviaciones típicas de la varianza que recoge su componente.

### Formulación

Con los datos centrados, las componentes son los **autovectores** $\mathbf{w}_m$ de la matriz de covarianzas $\mathbf{S}$, y cada **autovalor** $\lambda_m$ es la varianza que recoge su componente:

$$\mathbf{S}\,\mathbf{w}_m = \lambda_m \mathbf{w}_m, \qquad \text{fracción de varianza explicada} = \frac{\lambda_m}{\lambda_1 + \lambda_2 + \dots + \lambda_D}$$

✏️ **Rellena el hueco:** escribe la matriz de covarianzas `[[3, 3], [3, 4]]` (varianzas 3 y 4, covarianza 3).

In [ ]:
rng = np.random.default_rng(0)
cov = ___   # ✏️  matriz de covarianzas de las dos variables
X2d = rng.multivariate_normal(mean=[0, 0], cov=cov, size=500)

pca2d = PCA().fit(X2d)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X2d[:, 0], X2d[:, 1], s=12, alpha=0.3, color=AZUL)
for m, color in zip(range(2), [ROJO, NARANJA]):
    v = pca2d.components_[m] * 2 * np.sqrt(pca2d.explained_variance_[m])
    ax.plot([0, v[0]], [0, v[1]], lw=5, color=color,
            label=f"CP{m+1}: {pca2d.explained_variance_ratio_[m]*100:.0f} % de la varianza")
ax.set(aspect="equal", xlabel="x", ylabel="y")
ax.legend(); estilo(ax); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Qué porcentaje de la varianza recoge la CP1? ¿Por qué la CP2 es perpendicular a la CP1?
>
> 2) Cambia `cov` por `[[3, 0], [0, 3]]` (variables **no** correlacionadas) y vuelve a ejecutar. ¿Qué porcentaje recoge ahora la CP1? ¿Serviría el PCA para resumir estos datos?

---
## 2. Los datos de nivel del mar (SLA)

La **anomalía del nivel del mar** (SLA, *sea level anomaly*) es la diferencia entre el nivel del mar y su nivel medio, medida por satélite. `shoreline.mat` contiene la matriz `SLA` (puntos × meses) y las coordenadas `lat` y `lon` de cada punto. Descartamos un punto con longitud mayor de 160°.

In [ ]:
# Si trabajas en local, cambia la ruta por la de tu ordenador
ruta = "/Users/beatrizperezdiaz/Library/CloudStorage/OneDrive-UNICAN/DOCENCIA/2026_2027/AnaliticaDeDatos_PersonalBPD/MaterialBea_2aParte/Codes/AnaliticaDeDatos-main/datos/clases/clase_3_pca_clustering/shoreline.mat"
data_mat = loadmat(ruta)

sla = data_mat["SLA"]        # puntos x meses (m)
lat = data_mat["lat"][0]
lon = data_mat["lon"][0]

s = np.where(lon < 160)[0]
sla, lat, lon = sla[s, :], lat[s], lon[s]

fechas = pd.date_range("2000-01-01", periods=sla.shape[1], freq="MS")   # un registro al mes
print("Puntos de costa:", sla.shape[0], "| Meses:", sla.shape[1],
      "|", fechas[0].strftime("%m/%Y"), "a", fechas[-1].strftime("%m/%Y"))

Una función para dibujar cualquier valor en los puntos de costa (la usaremos varias veces):

In [ ]:
def dibuja_mapa(valores, titulo="", vlim=None, cmap="RdBu_r", ax=None, etiqueta="SLA (m)"):
    """Dibuja un valor por punto de costa sobre un mapa."""
    if ax is None:
        plt.figure(figsize=(14, 4))
        ax = plt.subplot(111, projection=ccrs.PlateCarree())
    ax.coastlines(zorder=2, color="navy")
    ax.gridlines(draw_labels=True, linewidth=1, color="lightgrey", linestyle=":", alpha=.5)
    vlim = vlim if vlim is not None else np.max(np.abs(valores))
    im = ax.scatter(lon, lat, s=50, c=valores, vmin=-vlim, vmax=vlim, cmap=cmap,
                    transform=ccrs.PlateCarree())
    plt.colorbar(im, ax=ax).set_label(etiqueta)
    ax.set_title(titulo)
    return ax

✏️ **Rellena el hueco:** elige un mes para ver su mapa de SLA (el índice va de 0 a 191; prueba `190`, noviembre de 2015).

In [ ]:
instante = ___   # ✏️
dibuja_mapa(sla[:, instante], titulo=f"SLA en {fechas[instante].strftime('%m/%Y')}")
plt.show()

### La matriz para el PCA

Cada **fila** es un mes (un mapa) y cada **columna** un punto de costa: una matriz de 192 × 171. **Estandarizamos** cada punto para que todos pesen lo mismo:

$$z_{t,j} = \frac{x_{t,j} - \bar x_j}{s_j} \qquad \text{(media y desviación típica de cada punto } j \text{)}$$

✏️ **Rellena el hueco:** la desviación típica de cada punto (columna) se calcula con `matriz.std(axis=0)`.

In [ ]:
matriz = sla.T                          # meses x puntos
medias = matriz.mean(axis=0)
desv = ___               # ✏️
matriz_norm = (matriz - medias) / desv
print("Matriz para el PCA:", matriz_norm.shape)

---
## 3. PCA: ¿cuántas componentes?

Ajustamos el PCA con todas las componentes y miramos la **varianza explicada acumulada**.

✏️ **Rellena el hueco:** elige el porcentaje de varianza que quieres conservar (empieza por `90`).

In [ ]:
pca = PCA().fit(matriz_norm)
PCs = pca.transform(matriz_norm)        # series temporales (meses x componentes)
EOFs = pca.components_.T                # patrones espaciales (puntos x componentes)
var_acum = np.cumsum(pca.explained_variance_ratio_) * 100

In [ ]:
n_perc = ___   # ✏️
n_pcs = int(np.argmax(var_acum >= n_perc)) + 1
print(f"Componentes para el {n_perc} % de la varianza: {n_pcs} (de {len(var_acum)})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(1, len(var_acum) + 1), var_acum, color=AZUL, lw=2)
ax.axvline(n_pcs, color=ROJO, ls="--", label=f"{n_pcs} componentes")
ax.axhline(n_perc, color="grey", ls=":")
ax.set_xlabel("Número de componentes"); ax.set_ylabel("% de varianza acumulada")
ax.legend(); estilo(ax); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Cuántas componentes hacen falta para el 90 %? ¿Y para el 99 %?
>
> 2) ¿Qué porcentaje recoge solo la primera componente? ¿Son muy redundantes las 171 series?

---
## 4. Patrones espaciales (EOF) y series temporales (PC)

Cuando las variables son puntos de un mapa, a los autovectores se les llama **EOF** (patrones espaciales) y a las puntuaciones, **PC** (series temporales). Puntos con el **mismo signo** en un EOF suben y bajan **a la vez**; signos opuestos indican un **contraste**. El signo global es arbitrario: lo que tiene sentido es el producto EOF × PC.

✏️ **Rellena el hueco:** empieza por la primera componente (`m = 0`) y después prueba `1` y `2`.

In [ ]:
m = ___   # ✏️

fig = plt.figure(figsize=(15, 6), tight_layout=True)
gs = gridspec.GridSpec(3, 1)
ax = fig.add_subplot(gs[:2, 0], projection=ccrs.PlateCarree())
dibuja_mapa(EOFs[:, m], titulo=f"EOF{m+1} ({pca.explained_variance_ratio_[m]*100:.1f} % de la varianza)",
            ax=ax, cmap="RdBu_r", etiqueta="Carga")
ax1 = fig.add_subplot(gs[2, 0])
ax1.plot(fechas, PCs[:, m], color=AZUL); ax1.axhline(0, color="grey", lw=0.8)
ax1.set_xlim(fechas[0], fechas[-1]); ax1.set_ylabel(f"PC{m+1}"); estilo(ax1)
plt.show()

> ### 💡 Preguntas
>
> 1) Mira la PC1: ¿cada cuántos meses se repite? ¿En qué época del año es máxima?
>
> 2) ¿Qué zonas tienen el mismo signo en el EOF1? ¿Qué fenómeno crees que representa?
>
> 3) ¿Se ve alguna tendencia en la PC1 a lo largo de los 16 años?

---
## 5. Reconstruir los mapas con pocas componentes

Con las `n_pcs` primeras componentes reconstruimos cada mapa y deshacemos la estandarización:

$$X(s, t) \approx \bar X(s) + s(s)\left[\text{EOF}_1(s)\,\text{PC}_1(t) + \dots + \text{EOF}_M(s)\,\text{PC}_M(t)\right]$$

El error que cometemos es la varianza de las componentes que dejamos fuera: $\sum_{m > M} \lambda_m$.

✏️ **Rellena el hueco:** para reconstruir multiplicamos las PC por los EOF **traspuestos** (`EOFs[:, :n_pcs].T`).

In [ ]:
reconstruido = (PCs[:, :n_pcs] @ ___) * desv + medias   # ✏️  meses x puntos
t = instante
error = reconstruido[t] - matriz[t]

fig = plt.figure(figsize=(15, 12), tight_layout=True)
paneles = [(matriz[t], f"Real ({fechas[t].strftime('%m/%Y')})", 0.3),
           (reconstruido[t], f"Reconstruido con {n_pcs} componentes", 0.3),
           (error, "Reconstruido − real", 0.04)]
for i, (valores, titulo, vlim) in enumerate(paneles):
    ax = fig.add_subplot(3, 1, i + 1, projection=ccrs.PlateCarree())
    dibuja_mapa(valores, titulo=titulo, vlim=vlim, ax=ax)
plt.show()

rmse_rec = np.sqrt(np.mean((reconstruido - matriz) ** 2)) * 100
print(f"Error cuadrático medio de la reconstrucción (todos los meses y puntos): {rmse_rec:.1f} cm")

> ### 💡 Preguntas
>
> 1) ¿Se parece el mapa reconstruido al real? ¿Dónde están las mayores diferencias?
>
> 2) Vuelve al bloque 3 y repite con `n_perc = 50` y con `n_perc = 99`. ¿Cómo cambian el mapa de diferencias y el error?

---
## 6. K-means sobre las componentes: ¿cuántos grupos?

Agrupamos los **meses** (los mapas) según sus coordenadas en el espacio de las componentes principales:

- K-means usa **todas** las componentes seleccionadas (`n_pcs`), no solo PC1 y PC2.
- **No** volvemos a estandarizar las PC: así cada una conserva su peso según la varianza que explica (la PC1 cuenta más que la PC20, igual que en el mapa original).
- No damos por hecho que haya grupos separados: lo comprobamos con el **codo** y la **silueta**.

### Formulación

K-means busca los $k$ centroides $\mathbf{v}_j$ que minimizan la **inercia** (suma de distancias al cuadrado de cada mes a su centroide):

$$J = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \left\lVert \mathbf{x}_i - \mathbf{v}_j \right\rVert^2$$

La **silueta** de cada punto compara la distancia media a su grupo ($a$) con la distancia al grupo vecino ($b$): $s = \dfrac{b - a}{\max(a, b)}$. Va de −1 a 1; una media por encima de 0,5 indica grupos claros y por debajo de 0,25, estructura débil.

✏️ **Rellena el hueco:** dentro del bucle, el número de grupos es `k`.

In [ ]:
Z = PCs[:, :n_pcs]        # todas las componentes seleccionadas, sin re-estandarizar

ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=___, n_init=10, random_state=0).fit(Z)   # ✏️
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Z, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inercias, "o-", color=AZUL, lw=2); axes[0].set_title("Método del codo")
axes[0].set_ylabel("Inercia J")
axes[1].plot(ks, siluetas, "o-", color=ROJO, lw=2); axes[1].set_title("Silueta media")
axes[1].axhspan(0, 0.25, color="grey", alpha=0.1)
for ax in axes:
    ax.set_xlabel("Número de grupos k"); estilo(ax)
plt.tight_layout(); plt.show()

> ### 💡 Preguntas
>
> 1) ¿Hay un codo claro en la inercia?
>
> 2) ¿Qué k sugiere la silueta? ¿Qué indica un valor en torno a 0,2-0,25?

✏️ **Rellena el hueco:** elige `k` (proponemos `4`; después prueba `3` y `5`).

In [ ]:
k = ___   # ✏️
km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
grupo = km.labels_
print("Meses en cada grupo:", np.bincount(grupo))

---
## 7. Interpretar los grupos

### 7.1. Los grupos en el plano PC1–PC2

Ojo: es solo una vista en dos dimensiones. K-means ha agrupado usando las `n_pcs` componentes, así que algunos grupos pueden solaparse en este plano.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for g in range(k):
    ax.scatter(Z[grupo == g, 0], Z[grupo == g, 1], s=25, color=COLORES_GRUPOS[g], label=f"Grupo {g}")
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], marker="X", s=220, c="k", label="Centroides")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(); estilo(ax); plt.show()

### 7.2. Mapa medio de cada grupo

Promediamos la anomalía de cada punto respecto a su nivel medio en los meses de cada grupo. Todos los mapas usan **la misma escala de colores**, para poder compararlos.

In [ ]:
anomalia = matriz - medias                                  # meses x puntos (m)
mapas = np.array([anomalia[grupo == g].mean(axis=0) for g in range(k)])
vlim = np.abs(mapas).max()

fig = plt.figure(figsize=(15, 4 * k), tight_layout=True)
for g in range(k):
    ax = fig.add_subplot(k, 1, g + 1, projection=ccrs.PlateCarree())
    dibuja_mapa(mapas[g], titulo=f"Grupo {g} ({np.sum(grupo == g)} meses)", vlim=vlim, ax=ax,
                etiqueta="Anomalía media (m)")
plt.show()

> ### 💡 Preguntas
>
> 1) Pon nombre a cada grupo: ¿subida generalizada, bajada generalizada, contraste entre zonas o hemisferios?

### 7.3. ¿Cuándo aparece cada situación?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4), gridspec_kw={"width_ratios": [1, 3, 1.5]})

axes[0].bar(range(k), np.bincount(grupo, minlength=k), color=COLORES_GRUPOS[:k])
axes[0].set_xlabel("Grupo"); axes[0].set_ylabel("Número de meses")

media_espacial = anomalia.mean(axis=1)
axes[1].plot(fechas, media_espacial, color="lightgrey", zorder=1)
axes[1].scatter(fechas, media_espacial, c=[COLORES_GRUPOS[g] for g in grupo], s=25, zorder=2)
axes[1].set_ylabel("Anomalía media de todos los puntos (m)")

for g in range(k):
    axes[2].bar(np.arange(12) + g * 0.2, np.bincount(fechas.month[grupo == g] - 1, minlength=12),
                width=0.2, color=COLORES_GRUPOS[g])
axes[2].set_xticks(np.arange(12) + 0.3, MESES, fontsize=8); axes[2].set_ylabel("Número de meses")

for ax in axes:
    estilo(ax)
plt.tight_layout(); plt.show()

> ### 💡 Preguntas
>
> 1) ¿En qué meses aparece cada grupo?
>
> 2) ¿Aparecen todos los años, o hay alguno que se concentre en una época del registro?
>
> 3) K-means no sabía qué mes era cada mapa. ¿Qué ha encontrado entonces?

---
## 8. *Ampliación (si sobra tiempo):* quitar el ciclo estacional y la tendencia

Lo que agrupa k-means depende de lo que dejemos en los datos. Quitamos a cada punto su **ciclo estacional** (la media de cada mes del año) y su **tendencia lineal**, y repetimos el PCA. Lo que queda es la variabilidad **de un año a otro**.

In [ ]:
t_meses = np.arange(len(fechas))
clima = np.array([anomalia[fechas.month == mes].mean(axis=0) for mes in range(1, 13)])
sin_ciclo = anomalia - clima[fechas.month - 1]               # quitamos el ciclo estacional
coef = np.polyfit(t_meses, sin_ciclo, 1)                      # tendencia lineal de cada punto
residuo = sin_ciclo - (np.outer(t_meses, coef[0]) + coef[1])  # quitamos la tendencia

print(f"Tendencia media: {coef[0].mean() * 12 * 1000:.1f} mm/año")

res_norm = residuo / residuo.std(axis=0)
pca_res = PCA().fit(res_norm)
PCs_res = pca_res.transform(res_norm)

fig = plt.figure(figsize=(15, 6), tight_layout=True)
gs = gridspec.GridSpec(3, 1)
ax = fig.add_subplot(gs[:2, 0], projection=ccrs.PlateCarree())
dibuja_mapa(pca_res.components_[0], titulo=f"EOF1 sin ciclo ni tendencia ({pca_res.explained_variance_ratio_[0]*100:.1f} %)",
            ax=ax, etiqueta="Carga")
ax1 = fig.add_subplot(gs[2, 0])
ax1.plot(fechas, PCs_res[:, 0], color=AZUL); ax1.axhline(0, color="grey", lw=0.8)
ax1.set_xlim(fechas[0], fechas[-1]); ax1.set_ylabel("PC1"); estilo(ax1)
plt.show()

> ### 💡 Preguntas
>
> 1) ¿Dónde son mayores las cargas del nuevo EOF1? ¿En qué años es la PC1 claramente positiva? ¿Os suena algún fenómeno?
>
> 2) Si repitieras el k-means con estos datos, ¿qué esperarías encontrar?

---
### 📌 Para llevarse

- **El PCA resume:** 171 series de nivel del mar se describen con 28 componentes (90 % de la varianza).
- **EOF y PC:** cada componente es un patrón espacial por una serie temporal. Mismo signo = suben a la vez; signos opuestos = contraste.
- **Reconstruir:** con pocas componentes se recupera casi todo; el error es la varianza que dejamos fuera.
- **K-means sobre las PC:** con todas las componentes seleccionadas y sin re-estandarizar, para respetar su importancia.
- **El codo y la silueta orientan, pero no deciden:** con una estructura débil, k se elige por interpretación.
- **Lo que agrupa k-means depende de los datos que le damos:** con el ciclo estacional encuentra estaciones; sin él, épocas o fases de El Niño.